In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.read.format("delta").table("workspace.default.caso_practico_ventas")
df.display()

In [0]:
df = df.toPandas()
print("=== Primeras 10 filas ===")
display(df.head(10))
print("=== Últimas 5 filas ===")
display(df.tail(5))
 
print(f"\nRESPUESTA: El dataset tiene {df.shape[0]} registros y {df.shape[1]} columnas.")
 
# ======================================================================
# ☐ 1.2 Identificación de Tipos
# ======================================================================
print("Tipos asignados automáticamente por Pandas:\n")
print(df.dtypes)
print()
df.info()
 
print("""
RESPUESTA: NO coinciden del todo con los tipos esperados:
 - fecha: se cargó como texto; debería ser fecha (datetime).
 - cantidad: se cargó como decimal (float) por los NaN; debería ser entero.
 - satisfaccion_cliente: se cargó como decimal (float) por los NaN; debería ser entero.
 - precio_unitario: float, coincide con lo esperado.
 - id_transaccion, id_cliente, categoria_producto, metodo_pago y region: texto, coinciden.
""")
 
# ======================================================================
# ☐ 1.3 Detección de Nulos
# ======================================================================
nulos = df.isnull().sum()
tabla_nulos = pd.DataFrame({
    "columna": nulos.index,
    "nulos": nulos.values,
    "porcentaje_%": (nulos.values / len(df) * 100).round(2)
})
display(tabla_nulos)
 
cols_con_nulos = tabla_nulos[tabla_nulos["nulos"] > 0]
print(f"RESPUESTA: Hay {int(nulos.sum())} valores nulos en total, distribuidos en {len(cols_con_nulos)} columnas:")
for _, f in cols_con_nulos.iterrows():
    print(f"   - {f['columna']}: {int(f['nulos'])} nulos ({f['porcentaje_%']}%)")
 
# ======================================================================
# ☐ 1.4 Estadística Descriptiva
# ======================================================================
estad = df[["cantidad", "precio_unitario"]].agg(["mean", "median", "std", "min", "max"]).round(2)
display(estad.reset_index().rename(columns={"index": "estadistico"}))
 
print(f"""
RESPUESTA: En precio_unitario la media ({estad.loc['mean','precio_unitario']}) es muy superior a la mediana
({estad.loc['median','precio_unitario']}) y la desviación estándar es muy alta ({estad.loc['std','precio_unitario']}).
El máximo ({estad.loc['max','precio_unitario']}) indica un valor atípico (outlier) que distorsiona la media.
En cantidad, la media ({estad.loc['mean','cantidad']}) y la mediana ({estad.loc['median','cantidad']}) son similares: distribución estable.
""")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC # BLOQUE 2 | FASE 2: Limpieza y Depuración (Data Cleaning)
 
# COMMAND ----------
 
# ======================================================================
# ☐ 2.1 Tratamiento de Duplicados
# ======================================================================
n_antes = len(df)
dup_completas = df.duplicated().sum()
dup_id = df["id_transaccion"].duplicated().sum()
 
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["id_transaccion"])
 
print(f"RESPUESTA: Se encontraron {dup_completas} filas completamente duplicadas y {dup_id} id_transaccion repetidos.")
print(f"Se eliminaron {n_antes - len(df)} registros: {n_antes} -> {len(df)} filas.")
 
# ======================================================================
# ☐ 2.2 Normalización de Texto (metodo_pago)
# ======================================================================
print("ANTES:")
print(df["metodo_pago"].value_counts())
 
df["metodo_pago"] = df["metodo_pago"].replace({"Tarjeta Credito": "Tarjeta de Crédito"})
 
print("\nDESPUÉS:")
print(df["metodo_pago"].value_counts())
print("\nRESPUESTA: Las variantes 'Tarjeta Credito' se unificaron como 'Tarjeta de Crédito'.")
 
# ======================================================================
# ☐ 2.3 Gestión de Valores Nulos
# ======================================================================
nulos_cant = df["cantidad"].isnull().sum()
nulos_sat = df["satisfaccion_cliente"].isnull().sum()
 
mediana_cant = df["cantidad"].median()
moda_sat = df["satisfaccion_cliente"].mode()[0]
 
df["cantidad"] = df["cantidad"].fillna(mediana_cant)
df["satisfaccion_cliente"] = df["satisfaccion_cliente"].fillna(moda_sat)
 
print(f"RESPUESTA:")
print(f" - cantidad: {nulos_cant} nulos reemplazados por la mediana = {mediana_cant}")
print(f" - satisfaccion_cliente: {nulos_sat} nulos imputados con la moda = {moda_sat}")
print("\nNulos restantes:")
print(df.isnull().sum())
 
# ======================================================================
# ☐ 2.4 Tratamiento de Atípicos (Outliers)
# ======================================================================
mask_outlier = df["precio_unitario"] == 9999.00
print("Transacción con precio_unitario = 9999.00:")
display(df[mask_outlier])
 
categoria_out = df.loc[mask_outlier, "categoria_producto"].iloc[0]
precio_medio_cat = df.loc[
    (df["categoria_producto"] == categoria_out) & ~mask_outlier, "precio_unitario"
].mean()
 
df.loc[mask_outlier, "precio_unitario"] = round(precio_medio_cat, 2)
 
print(f"""
RESPUESTA: Se decidió IMPUTAR el precio en lugar de eliminar la fila.
La transacción pertenece a la categoría '{categoria_out}' y se reemplazó el precio 9999.00 por el
precio medio de esa categoría ({precio_medio_cat:.2f}), calculado sin el valor atípico.
Justificación: 9999.00 es un precio irreal frente al resto de la categoría y se conserva
el resto de la información de la venta (cliente, fecha, región, satisfacción).
""")
 
# Convertir a los tipos correctos (ya no hay nulos en estas columnas)
df["cantidad"] = df["cantidad"].astype(int)
df["satisfaccion_cliente"] = df["satisfaccion_cliente"].astype(int)
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC # BLOQUE 3 | FASE 3: Transformación e Ingeniería de Atributos
 
# COMMAND ----------
 
# ======================================================================
# ☐ 3.1 Columna Derivada (monto_total)
# ======================================================================
df["monto_total"] = df["cantidad"] * df["precio_unitario"]
display(df[["id_transaccion", "cantidad", "precio_unitario", "monto_total"]].head(10))
 
print("RESPUESTA: Se creó la columna monto_total = cantidad * precio_unitario.")
print(f"Nota: {df['monto_total'].isnull().sum()} filas quedan con monto_total vacío porque su precio_unitario original era nulo.")
 
# ======================================================================
# ☐ 3.2 Conversión Temporal
# ======================================================================
df["fecha"] = pd.to_datetime(df["fecha"])
df["dia_semana"] = df["fecha"].dt.day_name()
df["mes"] = df["fecha"].dt.month
 
print(df.dtypes[["fecha"]])
display(df[["fecha", "dia_semana", "mes"]].head(10))
print("RESPUESTA: 'fecha' ahora es tipo datetime y se crearon las columnas 'dia_semana' y 'mes'.")
 
# ======================================================================
# ☐ 3.3 Discretización (Binning)
# ======================================================================
# Bajo: < 100 | Medio: 100 a 500 | Alto: > 500
df["rango_monto"] = pd.cut(
    df["monto_total"],
    bins=[-float("inf"), 99.9999, 500, float("inf")],
    labels=["Bajo", "Medio", "Alto"]
)
 
conteo = df["rango_monto"].value_counts().reindex(["Bajo", "Medio", "Alto"])
display(conteo.reset_index().rename(columns={"index": "rango", "rango_monto": "rango", "count": "transacciones"}))
 
print(f"RESPUESTA: Bajo (< $100): {conteo['Bajo']} | Medio ($100 a $500): {conteo['Medio']} | Alto (> $500): {conteo['Alto']} transacciones.")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC # BLOQUE 4 | FASE 4: Agregación y Resumen de Datos
 
# COMMAND ----------
 
# ======================================================================
# ☐ 4.1 Agrupamiento Simple (Group By)
# ======================================================================
resumen_cat = df.groupby("categoria_producto").agg(
    ingresos_totales=("monto_total", "sum"),
    unidades_promedio=("cantidad", "mean")
).round(2)
display(resumen_cat.reset_index())
 
top_ing = resumen_cat["ingresos_totales"].idxmax()
top_uni = resumen_cat["unidades_promedio"].idxmax()
print(f"RESPUESTA: La categoría con mayores ingresos es '{top_ing}' (${resumen_cat.loc[top_ing,'ingresos_totales']:,.2f}).")
print(f"La categoría con mayor promedio de unidades por venta es '{top_uni}' ({resumen_cat.loc[top_uni,'unidades_promedio']:.2f}).")
 
# ======================================================================
# ☐ 4.2 Tabla Dinámica (Pivot Table)
# ======================================================================
pivot = pd.pivot_table(
    df, values="monto_total", index="region",
    columns="categoria_producto", aggfunc="sum"
).round(2)
display(pivot.reset_index())
 
celda_max = pivot.stack().idxmax()
print(f"RESPUESTA: La combinación con mayores ingresos es región '{celda_max[0]}' en la categoría '{celda_max[1]}' (${pivot.stack().max():,.2f}).")
print(f"La región con más ingresos totales es '{pivot.sum(axis=1).idxmax()}'.")
 
# ======================================================================
# ☐ 4.3 Análisis de Satisfacción por Método de Pago
# ======================================================================
sat = df.groupby("metodo_pago")["satisfaccion_cliente"].mean().round(2)
display(sat.reset_index())
 
print(f"RESPUESTA: El método de pago con mayor satisfacción media es '{sat.idxmax()}' ({sat.max()}) y el menor es '{sat.idxmin()}' ({sat.min()}).")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC # BLOQUE 5 | FASE 5: Selección, Filtrado y Muestreo
 
# COMMAND ----------
 
# ======================================================================
# ☐ 5.1 Filtrado Multicriterio
# ======================================================================
filtro = df[(df["region"] == "Norte") & (df["satisfaccion_cliente"] >= 4)]
display(filtro)
 
print(f"RESPUESTA: Hay {len(filtro)} ventas de la región Norte con satisfacción igual o superior a 4.")
 
# ======================================================================
# ☐ 5.2 Muestreo Estratificado (10% por región)
# ======================================================================
muestra = df.groupby("region").sample(frac=0.10, random_state=42)
display(muestra)
 
comparacion = pd.DataFrame({
    "proporcion_poblacion_%": (df["region"].value_counts(normalize=True) * 100).round(1),
    "registros_muestra": muestra["region"].value_counts()
})
display(comparacion.reset_index().rename(columns={"index": "region", "region": "region"}))
 
print(f"RESPUESTA: Se extrajo una muestra de {len(muestra)} registros (aprox. 10% de {len(df)}), conservando la proporción de cada región.")
 